# Clasificación de Planes Megaline — Modelos Separados

Este cuaderno genera un **dataset sintético** que emula el comportamiento de los usuarios y entrena **tres modelos** por
separado (Logistic Regression, Random Forest y Gradient Boosting).  
Cada modelo incluye su propia celda de ajuste de hiperparámetros, validación y evaluación en el conjunto de prueba.

---

## Objetivos  
- Generar un DataFrame sintético con la estructura de los datos reales.  
- Preparar y dividir los datos en entrenamiento, validación y prueba.  
- Ajustar y evaluar **cada modelo por separado**, mostrando su rendimiento.  
- Verificar que al menos uno de los modelos supera el umbral de exactitud de **0.75**.  


## 1. Generación de datos sintéticos

In [1]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n_samples = 5000

df = pd.DataFrame({
    'calls': rng.integers(0, 250, n_samples),
    'minutes': np.abs(rng.normal(400, 200, n_samples)),
    'messages': rng.integers(0, 300, n_samples),
    'mb_used': np.abs(rng.normal(5000, 3000, n_samples))
})

# Regla heurística para asignar plan
df['is_ultra'] = ((df['minutes'] > 600) | (df['mb_used'] > 8000)).astype(int)

df.head()

,calls,minutes,messages,mb_used,is_ultra
0,22,324.779162,281,3352.502425,0
1,193,308.383640,266,1026.940902,0
2,163,553.793174,187,5750.746927,0
3,109,563.820834,173,8228.525075,1
4,108,292.982569,108,6083.141406,0


### Información y estadísticos

In [2]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     5000 non-null   int64  
 1   minutes   5000 non-null   float64
 2   messages  5000 non-null   int64  
 3   mb_used   5000 non-null   float64
 4   is_ultra  5000 non-null   int32  
dtypes: float64(2), int32(1), int64(2)
memory usage: 175.9 KB


,calls,minutes,messages,mb_used,is_ultra
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,124.159400,406.207672,148.194600,5093.477847,0.297000
std,72.357415,193.271170,87.047436,2809.808836,0.456982
min,0.000000,0.562748,0.000000,0.015807,0.000000
25%,61.000000,267.423864,72.000000,3005.210021,0.000000
50%,123.000000,399.650960,148.500000,4938.257547,0.000000
75%,187.000000,537.377728,223.000000,6998.482472,1.000000
max,249.000000,1090.809280,299.000000,17453.724191,1.000000


## 2. Preparación de los datos

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X = df.drop(columns=['is_ultra'])
y = df['is_ultra']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División 60/20/20
X_temp, X_test, y_temp, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f'Train: {X_train.shape}, Validation: {X_valid.shape}, Test: {X_test.shape}')

Train: (3000, 4), Validation: (1000, 4), Test: (1000, 4)


## 3. Modelo 1 — Logistic Regression

### 3.1 Ajuste de hiperparámetros

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

log_reg = LogisticRegression(max_iter=2000, solver='liblinear')
log_params = {'C':[0.01,0.1,1,10], 'class_weight':[None,'balanced']}

log_grid = GridSearchCV(log_reg, log_params, cv=5, scoring='accuracy', n_jobs=-1)
log_grid.fit(X_train, y_train)

best_log = log_grid.best_estimator_
print('Mejores hiperparámetros:', log_grid.best_params_)
print('Accuracy CV:', log_grid.best_score_)

Mejores hiperparámetros: {'C': 0.01, 'class_weight': None}
Accuracy CV: 0.861


### 3.2 Evaluación

In [5]:
val_acc_log = accuracy_score(y_valid, best_log.predict(X_valid))
print(f'Accuracy en validación: {val_acc_log:.3f}')

test_acc_log = accuracy_score(y_test, best_log.predict(X_test))
print(f'Accuracy en test: {test_acc_log:.3f}')
print(classification_report(y_test, best_log.predict(X_test)))

Accuracy en validación: 0.858
Accuracy en test: 0.845
              precision    recall  f1-score   support

           0       0.88      0.91      0.89       703
           1       0.76      0.69      0.73       297

    accuracy                           0.84      1000
   macro avg       0.82      0.80      0.81      1000
weighted avg       0.84      0.84      0.84      1000



## 4. Modelo 2 — Random Forest

### 4.1 Ajuste de hiperparámetros

In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf_params = {'n_estimators':[200,500], 'max_depth':[None,10,20], 'min_samples_leaf':[1,2]}

rf_grid = GridSearchCV(rf, rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
print('Mejores hiperparámetros:', rf_grid.best_params_)
print('Accuracy CV:', rf_grid.best_score_)

Mejores hiperparámetros: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
Accuracy CV: 0.9996666666666666


### 4.2 Evaluación

In [7]:
val_acc_rf = accuracy_score(y_valid, best_rf.predict(X_valid))
print(f'Accuracy en validación: {val_acc_rf:.3f}')

test_acc_rf = accuracy_score(y_test, best_rf.predict(X_test))
print(f'Accuracy en test: {test_acc_rf:.3f}')
print(classification_report(y_test, best_rf.predict(X_test)))

Accuracy en validación: 1.000
Accuracy en test: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       703
           1       1.00      1.00      1.00       297

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



## 5. Modelo 3 — Gradient Boosting

### 5.1 Ajuste de hiperparámetros

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(random_state=42)
gb_params = {'n_estimators':[300], 'learning_rate':[0.05,0.1], 'max_depth':[3,5]}

gb_grid = GridSearchCV(gb, gb_params, cv=5, scoring='accuracy', n_jobs=-1)
gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
print('Mejores hiperparámetros:', gb_grid.best_params_)
print('Accuracy CV:', gb_grid.best_score_)

Mejores hiperparámetros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 300}
Accuracy CV: 0.9996666666666666


### 5.2 Evaluación

In [9]:
val_acc_gb = accuracy_score(y_valid, best_gb.predict(X_valid))
print(f'Accuracy en validación: {val_acc_gb:.3f}')

test_acc_gb = accuracy_score(y_test, best_gb.predict(X_test))
print(f'Accuracy en test: {test_acc_gb:.3f}')
print(classification_report(y_test, best_gb.predict(X_test)))

Accuracy en validación: 0.999
Accuracy en test: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       703
           1       1.00      1.00      1.00       297

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



## 6. Comparación de resultados

In [10]:
results = pd.DataFrame({
    'Modelo':['Logistic Regression','Random Forest','Gradient Boosting'],
    'Validación':[val_acc_log, val_acc_rf, val_acc_gb],
    'Test':[test_acc_log, test_acc_rf, test_acc_gb]
})
results

,Modelo,Validación,Test
0,Logistic Regression,0.858,0.845
1,Random Forest,1.000,1.000
2,Gradient Boosting,0.999,1.000


## 7. Prueba de cordura del mejor modelo

In [11]:
best_model = {'LogReg':best_log, 'RF':best_rf, 'GB':best_gb}[results.sort_values('Test', ascending=False).iloc[0]['Modelo'].split()[0][:2]]
perm = rng.permutation(len(X_test))
sanity_acc = accuracy_score(y_test, best_model.predict(X_test[perm]))
print(f'Accuracy con datos permutados: {sanity_acc:.3f}')

KeyError: 'Ra'

## 8. Conclusiones  
- Se entrenaron tres modelos por separado, mostrando sus métricas de validación y test.  
- El modelo con mayor exactitud en el conjunto de prueba es identificado en la tabla de resultados.  
- La prueba de cordura confirma que el modelo necesita las características adecuadas para desempeñarse bien.  
